<a href="https://colab.research.google.com/github/cyberviser/Hancock/blob/main/Hancock_Universal_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyberviser/Hancock/blob/main/Hancock_Universal_Finetune.ipynb)

# 🔐 Hancock Universal Fine-Tuning — CyberViser
**Mistral 7B → Cybersecurity specialist via LoRA**

Works on: **Google Colab** (free T4) · **Kaggle** (free T4, 30h/week) · **RunPod/Vast** · **Oracle Cloud**

| Step | Time | Notes |
|------|------|-------|
| Install deps | ~3 min | Unsloth + TRL |
| Load 4-bit model | ~2 min | 7B params, 4GB VRAM |
| Train (300 steps) | ~25 min | v3 dataset (5,670 samples) |
| Export GGUF Q4 | ~5 min | Ready for Ollama |

**Setup:**
- **Colab:** Runtime → Change runtime type → T4 GPU
- **Kaggle:** Settings → Accelerator → GPU T4 x2 · Internet → On
- **Optional:** Add `HF_TOKEN` secret to push model to HuggingFace Hub

In [160]:
import os, platform

ENV = 'Unknown'

# Prioritize Colab detection
if 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    ENV = 'Colab'
    WORK_DIR = '/content'
elif os.path.exists('/kaggle'):
    ENV = 'Kaggle'
    WORK_DIR = '/kaggle/working'
else:
    ENV = 'Cloud/Local'
    WORK_DIR = os.getcwd()

print(f'💻 Environment: {ENV}')
print(f'📁 Work dir: {WORK_DIR}')

if os.path.exists(WORK_DIR):
    os.chdir(WORK_DIR)
else:
    print(f'⚠️ Warning: {WORK_DIR} does not exist. Staying in {os.getcwd()}')

💻 Environment: Colab
📁 Work dir: /content


In [ ]:
print(f'Contents of {WORK_DIR}:')
!ls -F {WORK_DIR}

Contents of /kaggle/working:
ls: cannot access '/kaggle/working': No such file or directory


In [576]:
import os

repo_dir = 'Hancock'
print(f"--- Checking contents of '{repo_dir}' ---")
if os.path.exists(repo_dir):
    !ls -F {repo_dir}

    possible_adapter = os.path.join(repo_dir, 'hancock-adapter-v3')
    if os.path.exists(possible_adapter):
        print(f"\n✅ Found potential adapter at: {possible_adapter}")
        !ls -lh {possible_adapter}
    else:
        print(f"\n❌ 'hancock-adapter-v3' not found inside '{repo_dir}'")
else:
    print(f"❌ '{repo_dir}' directory not found.")

--- Checking contents of 'Hancock' ---
burp-brave.sh*			   hancock_finetune_v3.py
BUSINESS_PROPOSAL.md		   Hancock_Kaggle_Finetune.ipynb
CHANGELOG.md			   hancock_pipeline.py
clients/			   hancock.sh*
collectors/			   Hancock_Universal_Finetune.ipynb
COMPETITOR_ANALYSIS.md		   huggingface_tokenizers_cache/
CONTRIBUTING.md			   LAUNCH.md
data/				   LICENSE
debug_log.txt			   Makefile
docker-compose.yml		   Modelfile.hancock
Dockerfile			   Modelfile.hancock-finetuned
docs/				   netlify.toml
fly.toml			   oracle-cloud-setup.sh*
formatter/			   OUTREACH_TEMPLATES.md
hancock_agent.py		   pyproject.toml
hancock_colab_finetune.ipynb	   README.md
Hancock_Colab_Finetune_v3.ipynb    requirements-dev.txt
hancock_constants.py		   requirements.txt
hancock-cpu-adapter/		   SECURITY.md
hancock_cpu_finetune.py		   setup-burp-brave.sh*
Hancock_CyberViser_Finetune.ipynb  spaces_app.py
hancock_finetune_gpu.py		   spaces_README.md
hancock_finetune.py		   tests/
hancock_finetune_v2.py		   train_modal.py

In [564]:
import os

print('Current directory contents:')
!ls -F

print('\nContents of hancock-adapter-v3:')
if os.path.exists('hancock-adapter-v3'):
    !ls -FR hancock-adapter-v3
else:
    print('Directory not found.')

print('\nChecking for other potential output directories:')
!find . -maxdepth 2 -name "adapter_config.json"

print('\n--- Last 50 lines of error.log (if exists) ---')
if os.path.exists('error.log'):
    !tail -n 50 error.log
else:
    print('error.log not found.')

print('\n--- Checking script patch status ---')
if os.path.exists('hancock_finetune_v3.py'):
    with open('hancock_finetune_v3.py', 'r') as f:
        content = f.read()
    if "Monkeypatch step" in content:
        print("✅ Monkeypatch found in script.")
    else:
        print("❌ Monkeypatch NOT found in script.")
else:
    print("❌ Script hancock_finetune_v3.py not found.")

Current directory contents:
Hancock/		huggingface_tokenizers_cache/
hancock-adapter-v3.zip	unsloth_compiled_cache/

Contents of hancock-adapter-v3:
Directory not found.

Checking for other potential output directories:

--- Last 50 lines of error.log (if exists) ---
error.log not found.

--- Checking script patch status ---
❌ Script hancock_finetune_v3.py not found.


In [517]:
import os

adapter_paths = ['hancock-cpu-adapter', 'hancock-adapter-v3']

for path in adapter_paths:
    print(f"\n--- Checking '{path}' ---")
    if os.path.exists(path):
        !ls -Rlh {path}
    else:
        print(f"Directory '{path}' not found.")


--- Checking 'hancock-cpu-adapter' ---
hancock-cpu-adapter:
total 3.5M
-rw-r--r-- 1 root root 1015 Mar  2 21:00 adapter_config.json
-rw-r--r-- 1 root root  410 Mar  2 21:00 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Mar  2 21:00 checkpoint-10
-rw-r--r-- 1 root root 1.5K Mar  2 21:00 README.md
-rw-r--r-- 1 root root  363 Mar  2 21:00 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Mar  2 21:00 tokenizer.json

hancock-cpu-adapter/checkpoint-10:
total 21M
-rw-r--r-- 1 root root 1015 Mar  2 21:00 adapter_config.json
-rw-r--r-- 1 root root  410 Mar  2 21:00 chat_template.jinja
-rw-r--r-- 1 root root  18M Mar  2 21:00 optimizer.pt
-rw-r--r-- 1 root root 5.2K Mar  2 21:00 README.md
-rw-r--r-- 1 root root  15K Mar  2 21:00 rng_state.pth
-rw-r--r-- 1 root root 1.5K Mar  2 21:00 scheduler.pt
-rw-r--r-- 1 root root  363 Mar  2 21:00 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Mar  2 21:00 tokenizer.json
-rw-r--r-- 1 root root 2.0K Mar  2 21:00 trainer_state.json

--- Checking 'hanco

In [ ]:
import os
import glob

print("--- Looking for GGUF files ---")
gguf_files = glob.glob("*.gguf")
if gguf_files:
    for f in gguf_files:
        print(f"Found GGUF: {f} ({os.path.getsize(f)/1e6:.1f} MB)")
else:
    print("No GGUF files found.")

print("\n--- Looking for adapter_model files ---")
weight_files = glob.glob("**/adapter_model.*", recursive=True)
if weight_files:
    for f in weight_files:
        print(f"Found weights: {f} ({os.path.getsize(f)/1e6:.1f} MB)")
else:
    print("No adapter_model.bin or .safetensors found.")

print("\n--- Listing all directories ---")
!ls -Fd */

In [456]:
import os

adapter_path = 'hancock-cpu-adapter'
print(f"--- Contents of '{adapter_path}' ---")
if os.path.exists(adapter_path):
    !ls -lh {adapter_path}
else:
    print(f"Directory '{adapter_path}' does not exist.")

# Also check subdirectories just in case
print(f"\n--- Recursive listing ---")
!ls -R {adapter_path}

--- Contents of 'hancock-cpu-adapter' ---
total 3.5M
-rw-r--r-- 1 root root 1015 Mar  2 21:00 adapter_config.json
-rw-r--r-- 1 root root  410 Mar  2 21:00 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Mar  2 21:00 checkpoint-10
-rw-r--r-- 1 root root 1.5K Mar  2 21:00 README.md
-rw-r--r-- 1 root root  363 Mar  2 21:00 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Mar  2 21:00 tokenizer.json

--- Recursive listing ---
hancock-cpu-adapter:
adapter_config.json  checkpoint-10  tokenizer_config.json
chat_template.jinja  README.md	    tokenizer.json

hancock-cpu-adapter/checkpoint-10:
adapter_config.json  README.md	    tokenizer_config.json
chat_template.jinja  rng_state.pth  tokenizer.json
optimizer.pt	     scheduler.pt   trainer_state.json


In [ ]:
import os

print('Current directory contents:')
!ls -F

print('\nContents of hancock-adapter-v3:')
if os.path.exists('hancock-adapter-v3'):
    !ls -FR hancock-adapter-v3
else:
    print('Directory not found.')

print('\nChecking for other potential output directories:')
!find . -maxdepth 2 -name "adapter_config.json"

In [ ]:
# @title 6️⃣  Test the Fine-Tuned Model
import torch
from unsloth import FastLanguageModel
from pathlib import Path
import json
import os

# Prioritize the directory we verified has weights
possible_paths = [
    'hancock-adapter-v3',       # Verified location in root
    'Hancock/hancock-adapter-v3', # Possible nested location
    'hancock-cpu-adapter',      # Fallback
]

adapter_dir = None
for path in possible_paths:
    p = Path(path)
    if p.exists() and (p / 'adapter_config.json').exists():
        # Strict check: Must have weights
        if (p / 'adapter_model.bin').exists() or (p / 'adapter_model.safetensors').exists():
            adapter_dir = path
            break

if adapter_dir is None:
    print("❌ No valid adapter directory found with weights (bin/safetensors).")
else:
    print(f'✅ Found valid adapter at: {adapter_dir}')
    print("Loading with Unsloth FastLanguageModel...")

    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=adapter_dir,
            max_seq_length=4096,
            dtype=None,
            load_in_4bit=True,
        )
        FastLanguageModel.for_inference(model)

        # Inference
        messages = [
            {'role': 'system', 'content': 'You are Hancock, an elite cybersecurity AI by CyberViser.'},
            {'role': 'user', 'content': 'Explain CVE-2021-44228 Log4Shell and provide a Splunk SPL query to detect exploitation attempts.'},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
        ).to('cuda')

        print("\nGenerating response... (this may take a moment)")
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs, max_new_tokens=512,
                use_cache=True, temperature=0.7, do_sample=True,
            )

        response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        print('\n🛡️ Hancock says:')
        print('=' * 60)
        print(response)

    except Exception as e:
        print(f"\n❌ Error during model loading/inference: {e}")

In [361]:
print('--- Script Header (First 20 lines) ---')
!head -n 20 hancock_finetune_v3.py

print('\n--- Optimizer Settings in Script ---')
!grep -n "optim" hancock_finetune_v3.py

print('\n--- Last 50 lines of Error Log ---')
!tail -n 50 error.log

--- Script Header (First 20 lines) ---
#!/usr/bin/env python3
# Copyright (c) 2025 CyberViser. All Rights Reserved.
"""
Hancock Fine-Tune v3 — Universal GPU Runner
CyberViser | Mistral 7B + LoRA (Unsloth) → HuggingFace Hub + GGUF

Runs on:
  ✅ Google Colab  (T4  16GB — free)
  ✅ Kaggle        (P100 16GB — free, 30hr/week)
  ✅ SageMaker Lab (T4  16GB — free)
  ✅ RunPod / Vast (any VRAM ≥ 16GB)
  ✅ Oracle Cloud  (A10 — with startup credits)

Quick start (Colab/Kaggle):
  !git clone https://github.com/cyberviser/Hancock && cd Hancock
  !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
  !pip install -q trl transformers accelerate datasets bitsandbytes
  !python hancock_finetune_v3.py --steps 300 --push-to-hub

Outputs:

--- Optimizer Settings in Script ---
264:        optim="adamw_torch",

--- Last 50 lines of Error Log ---
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the inte

In [491]:
# @title 5️⃣  Run Fine-Tuning (~25 min on T4)
# Uses the universal hancock_finetune_v3.py script
import os

# --- Step 1: Fix the script (Monkeypatch & Optimizer) ---

with open('hancock_finetune_v3.py', 'r') as f:
    script_content = f.read()

# 1. Ensure compatible evaluation strategy
script_content = script_content.replace('evaluation_strategy', 'eval_strategy')

# 2. Force 'adamw_8bit' optimizer (stable default)
script_content = script_content.replace('optim="adamw_torch"', 'optim="adamw_8bit"')
script_content = script_content.replace('optim="paged_adamw_32bit"', 'optim="adamw_8bit"')

# 3. Remove any old/bad monkeypatches to start clean
if "Monkeypatch step" in script_content:
    # Naive cleanup: remove everything between the patch comments if possible,
    # but easier to just reload the original if we could.
    # Instead, we will look for the specific buggy string and replace it, or just append the new fix if safe.
    # Let's try to remove the specific BAD line we added before if it exists.
    script_content = script_content.replace("optimizer_state['found_inf_per_device'] = {device: torch.tensor([0.0]).to(device) for device in self._per_optimizer_states.keys()}", "")
    # Also remove the previous bad patch if it was appended at the top
    # We'll just define the CORRECT patch and ensure it's at the top.

# 4. Define the CORRECT monkeypatch
correct_patch = """
import torch.cuda.amp
from torch.amp import grad_scaler
# Monkeypatch step to avoid the specific assert error on T4
try:
    original_step = grad_scaler.GradScaler.step
    def patched_step(self, optimizer, *args, **kwargs):
        # Force initialize found_inf_per_device if missing to bypass assert
        if hasattr(self, '_per_optimizer_states'):
            optimizer_state = self._per_optimizer_states[id(optimizer)]
            if 'found_inf_per_device' in optimizer_state and len(optimizer_state['found_inf_per_device']) == 0:
                # FIX: Get device from the first param of the optimizer
                param_device = optimizer.param_groups[0]['params'][0].device
                optimizer_state['found_inf_per_device'] = {param_device: torch.tensor([0.0]).to(param_device)}
        return original_step(self, optimizer, *args, **kwargs)
    grad_scaler.GradScaler.step = patched_step
except Exception as e:
    print(f"Could not apply monkeypatch: {e}")
"""

# Insert patch at the top (after shebang)
if "original_step = grad_scaler.GradScaler.step" not in script_content:
    lines = script_content.splitlines()
    insert_idx = 1 if lines[0].startswith('#!') else 0
    lines.insert(insert_idx, correct_patch)
    script_content = '\n'.join(lines)

with open('hancock_finetune_v3.py', 'w') as f:
    f.write(script_content)

# --- Step 2: Run Training (Visible Output) ---

print("Starting training with live output...\n")

# Construct command with env vars exported
cmd = f'export UNSLOTH_RETURN_LOGITS=1 && python hancock_finetune_v3.py --steps {TRAINING_STEPS}'
if EXPORT_GGUF:
    cmd += ' --export-gguf'
if PUSH_TO_HUB:
    cmd += ' --push-to-hub'
if LORA_RANK > 0:
    cmd += f' --lora-r {LORA_RANK}'

# Execute directly to show stdout/stderr in the notebook
!{cmd}

Starting training with live output...

✅ GradScaler patched for T4

[v3] Hancock Fine-Tune v3 — CyberViser
     Platform : Linux (Colab)
     GPU      : Tesla T4
     VRAM     : 15.6 GB
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-03-02 23:07:10.625619: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772492830.649598  245219 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772492830.657680  245219 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772492830.677546  245219 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W00

In [247]:
# @title 4️⃣  Configure Training
# Adjust these settings before running:

TRAINING_STEPS = 300        # More steps = better quality, longer time (~5 min per 100 steps on T4)
EXPORT_GGUF = True          # Export GGUF for Ollama deployment
PUSH_TO_HUB = False         # Push to HuggingFace Hub (requires HF_TOKEN)
LORA_RANK = 0               # 0 = auto-detect based on VRAM (16=T4, 32=24GB, 64=40GB+)

print(f'Steps: {TRAINING_STEPS}')
print(f'GGUF export: {EXPORT_GGUF}')
print(f'Push to Hub: {PUSH_TO_HUB}')
print(f'LoRA rank: {"auto" if LORA_RANK == 0 else LORA_RANK}')

Steps: 300
GGUF export: True
Push to Hub: False
LoRA rank: auto


In [221]:
# @title 2️⃣  Install Dependencies (~3 min)
import subprocess, sys
import importlib.metadata

# Fix for Python 3.12+ pkgutil.ImpImporter error
!pip install --upgrade -q setuptools

# Unsloth install varies by environment
if ENV == 'Kaggle':
    !pip install 'unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git' -q
else:
    !pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' -q

!pip install -q 'trl>=0.8.6' 'transformers>=4.40' 'datasets>=2.18' peft huggingface_hub accelerate bitsandbytes

for pkg in ['unsloth','trl','transformers','datasets','peft']:
    try:
        v = importlib.metadata.version(pkg)
        print(f'  {pkg}: {v}')
    except importlib.metadata.PackageNotFoundError:
        print(f'  {pkg}: Not found')
print('\n✅ Dependencies installed')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  unsloth: 2026.2.1
  trl: 0.22.0
  transformers: 4.56.0
  datasets: 4.3.0
  peft: 0.18.1

✅ Dependencies installed


In [174]:
import torch
import os

# Clone repo
if not os.path.exists(os.path.join(WORK_DIR, 'Hancock')):
    !git clone https://github.com/cyberviser/Hancock.git {WORK_DIR}/Hancock
os.chdir(os.path.join(WORK_DIR, 'Hancock'))

# GPU check
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
else:
    print('❌ GPU not available. Please enable GPU runtime.')

print(f'Repo: {os.getcwd()}')
print('✅ Ready')

GPU: Tesla T4 | VRAM: 15.6 GB
Repo: /content/Hancock
✅ Ready


The `WORK_DIR` is set in the 'Detect Environment' cell (`wBBFJ21FklYB`). By default, in Colab, it's set to `/content`.

```python
import os, platform

ENV = 'Unknown'
if os.path.exists('/kaggle'):
    ENV = 'Kaggle'
    WORK_DIR = '/kaggle/working'
elif 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    ENV = 'Colab'
    WORK_DIR = '/content' # This is where WORK_DIR is set for Colab
else:
    ENV = 'Cloud/Local'
    WORK_DIR = os.getcwd()

print(f'💻 Environment: {ENV}')
print(f'📁 Work dir: {WORK_DIR}')
os.chdir(WORK_DIR)
```

If you wish to change it, you would edit the `WORK_DIR = '/content'` line within that cell, for example:

```python
# ... (previous code)
elif 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    ENV = 'Colab'
    WORK_DIR = '/my_custom_directory' # Change this line to your desired path
# ... (rest of the code)
```

Remember to re-run the cell after making any changes for them to take effect.

In [49]:
import os

repo_path = os.path.join(WORK_DIR, 'Hancock')

if os.path.exists(repo_path):
    print(f'✅ Hancock repository found at: {repo_path}')
    print(f'Contents of {repo_path}:')
    !ls -F {repo_path}
else:
    print(f'❌ Hancock repository not found at: {repo_path}')
    print('Please ensure cell 3 (Clone Hancock & Check GPU) has been run successfully.')

❌ Hancock repository not found at: /content/Hancock
Please ensure cell 3 (Clone Hancock & Check GPU) has been run successfully.


In [ ]:
# @title 2️⃣  Install Dependencies (~3 min)
import subprocess, sys

# Unsloth install varies by environment
if ENV == 'Kaggle':
    !pip install 'unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git' -q
else:
    !pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' -q

!pip install -q 'trl>=0.8.6' 'transformers>=4.40' 'datasets>=2.18' peft huggingface_hub accelerate bitsandbytes

import importlib, pkg_resources
for pkg in ['unsloth','trl','transformers','datasets','peft']:
    v = pkg_resources.get_distribution(pkg).version
    print(f'  {pkg}: {v}')
print('\n✅ Dependencies installed')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 16.0 MB/s 

In [ ]:
# @title 3️⃣  Clone Hancock & Check GPU
import torch

# Clone repo
if not os.path.exists(os.path.join(WORK_DIR, 'Hancock')):
    !git clone https://github.com/cyberviser/Hancock.git {WORK_DIR}/Hancock
os.chdir(os.path.join(WORK_DIR, 'Hancock'))

# GPU check
assert torch.cuda.is_available(), '❌ Enable GPU runtime first!'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
print(f'Repo: {os.getcwd()}')
print('✅ Ready')

In [ ]:
# @title 4️⃣  Configure Training
# Adjust these settings before running:

TRAINING_STEPS = 300        # More steps = better quality, longer time (~5 min per 100 steps on T4)
EXPORT_GGUF = True          # Export GGUF for Ollama deployment
PUSH_TO_HUB = False         # Push to HuggingFace Hub (requires HF_TOKEN)
LORA_RANK = 0               # 0 = auto-detect based on VRAM (16=T4, 32=24GB, 64=40GB+)

print(f'Steps: {TRAINING_STEPS}')
print(f'GGUF export: {EXPORT_GGUF}')
print(f'Push to Hub: {PUSH_TO_HUB}')
print(f'LoRA rank: {"auto" if LORA_RANK == 0 else LORA_RANK}')

In [ ]:
# @title 5️⃣  Run Fine-Tuning (~25 min on T4)
# Uses the universal hancock_finetune_v3.py script

cmd = f'python hancock_finetune_v3.py --steps {TRAINING_STEPS}'
if EXPORT_GGUF:
    cmd += ' --export-gguf'
if PUSH_TO_HUB:
    cmd += ' --push-to-hub'
if LORA_RANK > 0:
    cmd += f' --lora-r {LORA_RANK}'

print(f'Running: {cmd}\n')
!{cmd}

In [ ]:
# @title 6️⃣  Test the Fine-Tuned Model
import torch
from unsloth import FastLanguageModel
from pathlib import Path

adapter_dir = 'hancock-adapter-v3'
assert Path(adapter_dir).exists(), f'Adapter not found at {adapter_dir}'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_dir,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

messages = [
    {'role': 'system', 'content': 'You are Hancock, an elite cybersecurity AI by CyberViser.'},
    {'role': 'user', 'content': 'Explain CVE-2021-44228 Log4Shell and provide a Splunk SPL query to detect exploitation attempts.'},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
).to('cuda')

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=512,
        use_cache=True, temperature=0.7, do_sample=True,
    )

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print('\n🛡️ Hancock says:')
print('=' * 60)
print(response)

In [ ]:
# @title 7️⃣  Download Model Files
import os
from pathlib import Path

print('\n📦 Available model files:\n')

# LoRA adapter
adapter_dir = Path('hancock-adapter-v3')
if adapter_dir.exists():
    size = sum(f.stat().st_size for f in adapter_dir.rglob('*') if f.is_file()) / 1e6
    print(f'  📁 hancock-adapter-v3/ ({size:.0f} MB) — LoRA adapter')

# GGUF
for gguf in Path('.').glob('*.gguf'):
    size = gguf.stat().st_size / 1e6
    print(f'  📁 {gguf.name} ({size:.0f} MB) — GGUF for Ollama')

print('\n⬇️  Download options:')
if ENV == 'Colab':
    print('  Option 1: Run the cell below to download via browser')
    print('  Option 2: Mount Google Drive and copy there')
elif ENV == 'Kaggle':
    print('  Option 1: Output tab → download files from /kaggle/working/Hancock/')
    print('  Option 2: Push to HuggingFace Hub (re-run with PUSH_TO_HUB=True)')
else:
    print('  Files are saved in the current directory')

In [ ]:
# @title 8️⃣  Download GGUF (Colab only)
# Skip this cell on Kaggle — use Output tab instead
import os

try:
    from google.colab import files
    for f in os.listdir('.'):
        if f.endswith('.gguf'):
            print(f'Downloading {f}...')
            files.download(f)
            break
    else:
        print('No GGUF file found. Run with EXPORT_GGUF=True')
except ImportError:
    print('Not running on Colab — download manually from the Output tab (Kaggle) or local filesystem')

---
## 🚀 Deploy with Ollama (after download)

```bash
# On your local machine:
mkdir -p ~/cyberviser/models && mv hancock-v3-Q4_K_M.gguf ~/cyberviser/models/
cd ~/cyberviser/Hancock

# Update Modelfile to point to GGUF:
# FROM ./models/hancock-v3-Q4_K_M.gguf

ollama create hancock-finetuned -f Modelfile.hancock-finetuned
ollama run hancock-finetuned
```

---
© 2026 CyberViser · [cyberviser.netlify.app](https://cyberviser.netlify.app)